# 01 — Data Exploration

Visualise the raw and processed PBMC data, confirm QC metrics, inspect
cell-type label distribution, and sanity-check the tokenisation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 1
plt.rcParams['figure.dpi'] = 120

## 1. Load processed data

In [ ]:
adata = sc.read_h5ad('../data/processed.h5ad')
print(adata)
adata.obs.head()

## 2. QC metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

if 'n_genes_by_counts' not in adata.obs.columns:
    sc.pp.calculate_qc_metrics(adata, inplace=True)

axes[0].hist(adata.obs['n_genes_by_counts'], bins=50, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Genes per cell'); axes[0].set_title('Genes / cell')

axes[1].hist(adata.obs['total_counts'], bins=50, color='salmon', edgecolor='none')
axes[1].set_xlabel('Total counts'); axes[1].set_title('UMI counts / cell')

if 'pct_counts_mt' in adata.obs.columns:
    axes[2].hist(adata.obs['pct_counts_mt'], bins=50, color='seagreen', edgecolor='none')
    axes[2].set_xlabel('% MT counts'); axes[2].set_title('Mitochondrial %')
else:
    axes[2].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Cell-type label distribution

In [ ]:
ct_counts = adata.obs['cell_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ct_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_ylabel('Cell count')
ax.set_title('Cell-type distribution')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(ct_counts)

## 4. UMAP of raw scanpy embeddings (baseline)

In [ ]:
if 'X_umap' not in adata.obsm:
    sc.pp.pca(adata, n_comps=50)
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
    sc.tl.umap(adata)

sc.pl.umap(adata, color='cell_type', title='Scanpy UMAP (PCA baseline)', frameon=False)

## 5. Tokenisation sanity check

In [ ]:
from data.dataset import scRNADataset

ds = scRNADataset(adata, max_seq_len=256, mode='pretrain')
sample = ds[0]

print('Keys:', list(sample.keys()))
print('input_ids shape :', sample['input_ids'].shape)
print('attention_mask  :', sample['attention_mask'].sum().item(), 'real tokens')
print('labels (masked) :', (sample['labels'] != -100).sum().item(), 'positions masked')
print('First 20 input_ids:', sample['input_ids'][:20].tolist())

## 6. Expression rank distribution per cell

In [ ]:
import scipy.sparse as sp

X = adata.X
if sp.issparse(X):
    X = X.toarray()

expressed_per_cell = (X > 0).sum(axis=1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(expressed_per_cell, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel('Expressed HVGs per cell')
ax.set_title('Sequence length distribution (before truncation)')
ax.axvline(256, color='red', linestyle='--', label='max_seq_len=256 (demo)')
ax.axvline(2048, color='orange', linestyle='--', label='max_seq_len=2048 (full)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Median expressed HVGs: {np.median(expressed_per_cell):.0f}')
print(f'Max expressed HVGs:    {expressed_per_cell.max()}')